# ЛР 3

In [1]:
import numpy as np

In [2]:
import gensim.downloader
word2vec_rus = gensim.downloader.load('word2vec-ruscorpora-300')

## 1 задание

In [4]:
similar = word2vec_rus.most_similar('ключ_NOUN', topn=10)
print("Топ-10 похожих на 'ключ_NOUN':")
for word, score in similar:
    print(f"{word}: {score:.4f}")

Топ-10 похожих на 'ключ_NOUN':
отпирать_VERB: 0.6206
ключик_NOUN: 0.6106
отмычка_NOUN: 0.5797
замок_NOUN: 0.5736
отмыкать_VERB: 0.5663
запирать_VERB: 0.5614
врезной_ADJ: 0.5324
замочный::скважина_NOUN: 0.4879
ключий_ADJ: 0.4875
сезам::отворяться_VERB: 0.4853


## 2 задание

In [6]:
w1 = 'длинный_ADJ'
w2 = 'долгий_ADJ'
w3 = 'короткий_ADJ'

vec1 = word2vec_rus[w1]
vec2 = word2vec_rus[w2]
vec3 = word2vec_rus[w3]

sim12 = word2vec_rus.similarity(w1, w2)
sim13 = word2vec_rus.similarity(w1, w3)

print(f"Сходство между '{w1}' и '{w2}': {sim12:.4f}")
print(f"Сходство между '{w1}' и '{w3}': {sim13:.4f}")

if sim12 < sim13:
    print("Условие выполнено: сходство синонимов меньше, чем сходство с антонимом.")
else:
    print("Условие не выполнено, попробуйте другую тройку слов.")

Сходство между 'длинный_ADJ' и 'долгий_ADJ': 0.3873
Сходство между 'длинный_ADJ' и 'короткий_ADJ': 0.6513
Условие выполнено: сходство синонимов меньше, чем сходство с антонимом.


## Задание 3

In [8]:
import os
import urllib.request
import tarfile
import numpy as np

if not os.path.exists("aclImdb_v1.tar.gz"):
    print("Скачивание aclImdb_v1.tar.gz...")
    urllib.request.urlretrieve("https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz", "aclImdb_v1.tar.gz")
    with tarfile.open("aclImdb_v1.tar.gz", "r:gz") as tar:
        tar.extractall()
    print("Распаковано.")
else:
    print("Файл уже есть, распаковка не требуется.")

def preprocess(text):
    return text.lower()

def get_dataset(split='train', base_dir='aclImdb'):
    path_data = os.path.join(base_dir, split)
    path_neg = os.path.join(path_data, 'neg')
    path_pos = os.path.join(path_data, 'pos')
    
    neg_files = sorted(os.listdir(path_neg))
    pos_files = sorted(os.listdir(path_pos))
    
    X = []
    y = [0] * len(neg_files) + [1] * len(pos_files)
    
    for fname in neg_files:
        with open(os.path.join(path_neg, fname), 'r', encoding='utf-8') as f:
            X.append(preprocess(f.readline()))
    for fname in pos_files:
        with open(os.path.join(path_pos, fname), 'r', encoding='utf-8') as f:
            X.append(preprocess(f.readline()))
    
    return X, y

X_train, y_train = get_dataset('train')
X_test, y_test = get_dataset('test')
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

def get_corpus(data):
    return [x.split() for x in data]

corpus_train = get_corpus(X_train)
corpus_test  = get_corpus(X_test)

Файл уже есть, распаковка не требуется.
Train size: 25000, Test size: 25000


In [9]:
import fasttext

model_ft_char = fasttext.train_unsupervised(
    'all_data.txt',
    model='skipgram',
    dim=100,
    epoch=10,
    minn=3,
    maxn=6,
    verbose=1
)

model_ft_char.save_model('fasttext_char.bin')

In [10]:
def encode_corpus(corpus, model, vector_size=100):
    encoded = []
    for words in corpus:
        vecs = [model[word] for word in words if word in model]
        if vecs:
            encoded.append(np.mean(vecs, axis=0))
        else:
            encoded.append(np.zeros(vector_size))
    return np.array(encoded)

In [11]:
def get_corpus(data):
    return [x.split() for x in data]

corpus_train = get_corpus(X_train)
corpus_test  = get_corpus(X_test)

enc_train_char = encode_corpus(corpus_train, model_ft_char)
enc_test_char  = encode_corpus(corpus_test,  model_ft_char)
print("Размер обучающей матрицы:", enc_train_char.shape)
print("Размер тестовой матрицы:", enc_test_char.shape)

Размер обучающей матрицы: (25000, 100)
Размер тестовой матрицы: (25000, 100)


In [12]:
from catboost import CatBoostClassifier

clf_char = CatBoostClassifier(iterations=500, verbose=False, random_seed=42)
clf_char.fit(enc_train_char, y_train)

y_pred_char = clf_char.predict(enc_test_char)
accuracy = np.mean(y_pred_char.flatten() == y_test)
print(f"Точность на тесте (FastText + символьные n-граммы): {accuracy:.3f}")

Точность на тесте (FastText + символьные n-граммы): 0.877
